In [24]:
import pandas as pd
import os
from datetime import datetime, timedelta
directory = 'C:/Users/tuan-/Downloads/1 Lobster Thesis/Data/SPY2019/'

In [3]:
# Konverter tid
def convert_to_datetime(seconds, base_date):
    base_time = datetime.strptime(base_date, '%Y-%m-%d')
    return base_time + timedelta(seconds=seconds)

monthly_data = []

year = 2019

for month in range(1, 13):  # Loop monthly 
    monthly_files = []
    
    for filename in os.listdir(directory):
        if f'{year}-{month:02}' in filename and 'message' in filename:   # Match filer monthly and year
            message_file = os.path.join(directory, filename)
            orderbook_file = message_file.replace('message', 'orderbook')  # Match orderbook fil
            
             # Load message and orderbook 
            message_df = pd.read_csv(message_file)
            orderbook_df = pd.read_csv(orderbook_file)

            # Message and orderbook data, dropper col 7
            message_df = message_df.iloc[:, :-1]  
            message_df.columns = ['Time (sec)', 'Event Type', 'Order ID', 'Size', 'Price', 'Direction']
            orderbook_df.columns = ['Ask Price 1', 'Ask Size 1', 'Bid Price 1', 'Bid Size 1', 
                                    'Ask Price 2', 'Ask Size 2', 'Bid Price 2', 'Bid Size 2']

            base_date = filename.split('_')[1]  # Start dag SPY_2019-01-02
            message_df['Time (sec)'] = message_df['Time (sec)'].apply(lambda x: convert_to_datetime(x, base_date))

            # Merger message and orderbook data on 'Time (sec)'
            combined_df = pd.merge(orderbook_df, message_df, left_index=True, right_index=True, how='left')

            # Drop NAN
            combined_df.dropna(inplace=True)

            # Set 'Time (sec)' index for 1 second
            combined_df.set_index('Time (sec)', inplace=True)

            # Sampler data 1 sec interval første obs hvert sec
            resampled_df = combined_df.resample('1S').first()

            # list monthly
            monthly_files.append(resampled_df)
    
    # Concatenate all daily filer for month
    if monthly_files:
        monthly_data_df = pd.concat(monthly_files)
        monthly_data.append(monthly_data_df)
        
        # Gem
        monthly_data_df.to_csv(f'processed_{year}_{month:02}.csv', index=False)

# All months into one final DataFrame
final_df = pd.concat(monthly_data)

# Save the final DataFrame, year 2019
final_df.to_csv(f'combined_SPY{year}_cleaned.csv', index=False)


print(final_df.head())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2019-01-02 09:30:00    2460200.0       100.0    2459600.0       500.0   
2019-01-02 09:30:01    2460200.0      1200.0    2460100.0       200.0   
2019-01-02 09:30:02    2460400.0       500.0    2460200.0       300.0   
2019-01-02 09:30:03    2460300.0       100.0    2459800.0       100.0   
2019-01-02 09:30:04    2460100.0      1500.0    2459600.0      2400.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2019-01-02 09:30:00    2460400.0       500.0    2459300.0       500.0   
2019-01-02 09:30:01    2460600.0       500.0    2459800.0       500.0   
2019-01-02 09:30:02    2460500.0       200.0    2460000.0       500.0   
2019-01-02 09:30:03    2460400.0       500.0    2459300.0       500.0   
2019-01-02 09:30:04    2460200.0       124.0    24

In [4]:
print(final_df.head())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2019-01-02 09:30:00    2460200.0       100.0    2459600.0       500.0   
2019-01-02 09:30:01    2460200.0      1200.0    2460100.0       200.0   
2019-01-02 09:30:02    2460400.0       500.0    2460200.0       300.0   
2019-01-02 09:30:03    2460300.0       100.0    2459800.0       100.0   
2019-01-02 09:30:04    2460100.0      1500.0    2459600.0      2400.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2019-01-02 09:30:00    2460400.0       500.0    2459300.0       500.0   
2019-01-02 09:30:01    2460600.0       500.0    2459800.0       500.0   
2019-01-02 09:30:02    2460500.0       200.0    2460000.0       500.0   
2019-01-02 09:30:03    2460400.0       500.0    2459300.0       500.0   
2019-01-02 09:30:04    2460200.0       124.0    24

In [5]:
print(final_df.tail())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2019-12-31 15:59:55    3220000.0      1601.0    3219900.0       200.0   
2019-12-31 15:59:56    3220500.0      1800.0    3220400.0       155.0   
2019-12-31 15:59:57    3220500.0       301.0    3220400.0      4000.0   
2019-12-31 15:59:58    3220000.0      5205.0    3219900.0      2200.0   
2019-12-31 15:59:59    3220000.0       500.0    3219900.0       601.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2019-12-31 15:59:55    3220100.0      1301.0    3219800.0       501.0   
2019-12-31 15:59:56    3220600.0       901.0    3220300.0       101.0   
2019-12-31 15:59:57    3220600.0       901.0    3220300.0      1001.0   
2019-12-31 15:59:58    3220100.0       901.0    3219800.0      6301.0   
2019-12-31 15:59:59    3220100.0       901.0    32

In [15]:
# Observations (rows) in the final combined DataFrame for the year
total_observations_final = len(final_df)

print(f"Total observations in the final combined DataFrame: {total_observations_final}")

Total observations in the final combined DataFrame: 5835220


In [16]:
final_df.head()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction,Date
Time (sec),,,,,,,,,,,,,,
2019-01-02 09:30:00,2460200.0,100.0,2459600.0,500.0,2460400.0,500.0,2459300.0,500.0,1.0,11542096.0,100.0,2460200.0,-1.0,2019-01-02
2019-01-02 09:30:01,2460200.0,1200.0,2460100.0,200.0,2460600.0,500.0,2459800.0,500.0,1.0,11779468.0,500.0,2460200.0,-1.0,2019-01-02
2019-01-02 09:30:02,2460400.0,500.0,2460200.0,300.0,2460500.0,200.0,2460000.0,500.0,1.0,11925172.0,500.0,2460400.0,-1.0,2019-01-02
2019-01-02 09:30:03,2460300.0,100.0,2459800.0,100.0,2460400.0,500.0,2459300.0,500.0,3.0,12031688.0,30.0,2460000.0,1.0,2019-01-02
2019-01-02 09:30:04,2460100.0,1500.0,2459600.0,2400.0,2460200.0,124.0,2459300.0,500.0,1.0,12084756.0,2000.0,2459600.0,1.0,2019-01-02


In [17]:
final_df.tail()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction,Date
Time (sec),,,,,,,,,,,,,,
2019-12-31 15:59:55,3220000.0,1601.0,3219900.0,200.0,3220100.0,1301.0,3219800.0,501.0,1.0,284025740.0,200.0,3219900.0,1.0,2019-12-31
2019-12-31 15:59:56,3220500.0,1800.0,3220400.0,155.0,3220600.0,901.0,3220300.0,101.0,3.0,284109352.0,100.0,3220500.0,-1.0,2019-12-31
2019-12-31 15:59:57,3220500.0,301.0,3220400.0,4000.0,3220600.0,901.0,3220300.0,1001.0,1.0,284178968.0,2000.0,3220400.0,1.0,2019-12-31
2019-12-31 15:59:58,3220000.0,5205.0,3219900.0,2200.0,3220100.0,901.0,3219800.0,6301.0,1.0,284273000.0,2000.0,3219900.0,1.0,2019-12-31
2019-12-31 15:59:59,3220000.0,500.0,3219900.0,601.0,3220100.0,901.0,3219800.0,1601.0,1.0,284376636.0,500.0,3219900.0,1.0,2019-12-31


In [9]:
# Save 
final_df.to_csv('final_combined_2019.csv', index=True)  # Keep the index to retain 'Time (sec)'

final_df.to_csv('final_combined_2019_with_time.csv', index=True)

In [21]:
# Count unique days
final_df['Date'] = final_df.index.date

# Count the number of unique trading days
unique_trading_days = final_df['Date'].nunique()

print(f"Number of unique trading days: {unique_trading_days}")

Number of unique trading days: 252


In [26]:
# Load the data from 'final_combined_2019_with_time.csv'
final_combined_2019_with_time = pd.read_csv('final_combined_2019_with_time.csv')


final_combined_2019_with_time['Time (sec)'] = pd.to_datetime(final_combined_2019_with_time['Time (sec)'])

# Group trading days
grouped = final_combined_2019_with_time.groupby(final_combined_2019_with_time['Time (sec)'].dt.date)

# Sampler liste
resampled_5min_list = []

# Iterate 
for date, group in grouped:
    # Filtrer data for trading hours (9:30 AM to 4:00 PM)
    group_trading_hours = group[
        (group['Time (sec)'].dt.time >= pd.to_datetime('09:30:00').time()) &
        (group['Time (sec)'].dt.time <= pd.to_datetime('16:00:00').time())
    ]
    
    # Resample for 5-minute intervals 
    resampled_day = group_trading_hours.resample('5T', on='Time (sec)').agg({
        'Ask Price 1': ['first', 'max', 'min', 'last'],
        'Bid Price 1': ['first', 'max', 'min', 'last'],
        'Ask Price 2': ['first', 'max', 'min', 'last'],
        'Bid Price 2': ['first', 'max', 'min', 'last'],
        'Ask Size 1': ['sum', 'mean'],
        'Bid Size 1': ['sum', 'mean'],
        'Ask Size 2': ['sum', 'mean'],
        'Bid Size 2': ['sum', 'mean'],
        'Price': ['first', 'max', 'min', 'last'],
        'Direction': 'mean'
    })
    
   
    resampled_day.columns = ['_'.join(col).strip() for col in resampled_day.columns.values]
    
    # Append the resampled data dag
    resampled_5min_list.append(resampled_day)

# Concatenate all, DataFrame
resampled_5min_final_df = pd.concat(resampled_5min_list)

# Reset index 'Time (sec)' 
resampled_5min_final_df.reset_index(inplace=True)

In [27]:
print(resampled_5min_final_df.head(10))

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2019-01-02 09:30:00          2460200.0        2466000.0        2460100.0   
1 2019-01-02 09:35:00          2463000.0        2467400.0        2459700.0   
2 2019-01-02 09:40:00          2465300.0        2469200.0        2461500.0   
3 2019-01-02 09:45:00          2469100.0        2473200.0        2468900.0   
4 2019-01-02 09:50:00          2471300.0        2474000.0        2467400.0   
5 2019-01-02 09:55:00          2469200.0        2472600.0        2468600.0   
6 2019-01-02 10:00:00          2471900.0        2472100.0        2466200.0   
7 2019-01-02 10:05:00          2472100.0        2484500.0        2471800.0   
8 2019-01-02 10:10:00          2483300.0        2486000.0        2482200.0   
9 2019-01-02 10:15:00          2484000.0        2485600.0        2480300.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         2463200.0          2459600.0        2465800.0        24

In [28]:
# Save
resampled_5min_final_df.to_csv('resampled_5min_final_2019_corrected.csv', index=False)

In [29]:
# Count observations for each day in 5-minute interval
resampled_5min_final_df['Date'] = resampled_5min_final_df['Time (sec)'].dt.date

# Count the number of rows for each day
observations_per_day = resampled_5min_final_df.groupby('Date').size()


print(observations_per_day)

Date
2019-01-02    78
2019-01-03    78
2019-01-04    78
2019-01-07    78
2019-01-08    78
              ..
2019-12-24    42
2019-12-26    78
2019-12-27    78
2019-12-30    78
2019-12-31    78
Length: 252, dtype: int64


In [30]:
print(resampled_5min_final_df.head(10))

# Save igen
resampled_5min_final_df.to_csv('resampled_5min_final_2019_corrected.csv', index=False)

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2019-01-02 09:30:00          2460200.0        2466000.0        2460100.0   
1 2019-01-02 09:35:00          2463000.0        2467400.0        2459700.0   
2 2019-01-02 09:40:00          2465300.0        2469200.0        2461500.0   
3 2019-01-02 09:45:00          2469100.0        2473200.0        2468900.0   
4 2019-01-02 09:50:00          2471300.0        2474000.0        2467400.0   
5 2019-01-02 09:55:00          2469200.0        2472600.0        2468600.0   
6 2019-01-02 10:00:00          2471900.0        2472100.0        2466200.0   
7 2019-01-02 10:05:00          2472100.0        2484500.0        2471800.0   
8 2019-01-02 10:10:00          2483300.0        2486000.0        2482200.0   
9 2019-01-02 10:15:00          2484000.0        2485600.0        2480300.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         2463200.0          2459600.0        2465800.0        24

In [31]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.head(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2019_corrected.csv', index=False)

Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2019-01-02 09:30:00,2460200.0,2466000.0,2460100.0,2463200.0,2459600.0,2465800.0,2459600.0,2463100.0,2460400.0,2466200.0,2460200.0,2463300.0,2459300.0,2465700.0,2459300.0,2463000.0,98895.0,329.650000,125445.0,418.150000,139813.0,466.043333,146817.0,489.390000,2460200.0,2465800.0,2459600.0,2463200.0,-0.020000,2019-01-02
2019-01-02 09:35:00,2463000.0,2467400.0,2459700.0,2465200.0,2462900.0,2467200.0,2459600.0,2465100.0,2463100.0,2467500.0,2459800.0,2465300.0,2462800.0,2467100.0,2459500.0,2465000.0,118706.0,395.686667,136407.0,454.690000,178107.0,593.690000,208044.0,693.480000,2463000.0,2467200.0,2459500.0,2465100.0,-0.013333,2019-01-02
2019-01-02 09:40:00,2465300.0,2469200.0,2461500.0,2469000.0,2465200.0,2469100.0,2461400.0,2468800.0,2465400.0,2469300.0,2461600.0,2469100.0,2465100.0,2469000.0,2461300.0,2468700.0,121790.0,405.966667,134217.0,447.390000,169047.0,563.490000,175121.0,583.736667,2465200.0,2469100.0,2461500.0,2469000.0,0.040000,2019-01-02
2019-01-02 09:45:00,2469100.0,2473200.0,2468900.0,2471600.0,2469000.0,2473100.0,2468800.0,2471400.0,2469200.0,2473300.0,2469000.0,2471700.0,2468900.0,2473000.0,2468700.0,2471300.0,138454.0,461.513333,123182.0,410.606667,181419.0,604.730000,186565.0,621.883333,2469100.0,2473200.0,2468800.0,2471300.0,0.373333,2019-01-02
2019-01-02 09:50:00,2471300.0,2474000.0,2467400.0,2469300.0,2471200.0,2473900.0,2467300.0,2469100.0,2471400.0,2474100.0,2467500.0,2469400.0,2471100.0,2473800.0,2467200.0,2469000.0,174305.0,581.016667,144568.0,481.893333,264402.0,881.340000,194594.0,648.646667,2471300.0,2474000.0,2467300.0,2469000.0,-0.100000,2019-01-02
2019-01-02 09:55:00,2469200.0,2472600.0,2468600.0,2471700.0,2469100.0,2472500.0,2468500.0,2471500.0,2469300.0,2472700.0,2468700.0,2471800.0,2469000.0,2472400.0,2468400.0,2471400.0,127148.0,423.826667,149196.0,497.320000,202608.0,675.360000,182942.0,609.806667,2469200.0,2472500.0,2468500.0,2471700.0,-0.013333,2019-01-02
2019-01-02 10:00:00,2471900.0,2472100.0,2466200.0,2472100.0,2471800.0,2472000.0,2466100.0,2472000.0,2472000.0,2472200.0,2466300.0,2472200.0,2471700.0,2471900.0,2466000.0,2471900.0,173369.0,577.896667,167280.0,557.600000,270042.0,900.140000,236332.0,787.773333,2471900.0,2472100.0,2466200.0,2472100.0,-0.013333,2019-01-02
2019-01-02 10:05:00,2472100.0,2484500.0,2471800.0,2483400.0,2472000.0,2484300.0,2471700.0,2483300.0,2472200.0,2484600.0,2471900.0,2483500.0,2471900.0,2484200.0,2471600.0,2483200.0,148240.0,494.133333,142412.0,474.706667,212575.0,708.583333,194203.0,647.343333,2472100.0,2484300.0,2471700.0,2483300.0,-0.093333,2019-01-02
2019-01-02 10:10:00,2483300.0,2486000.0,2482200.0,2483900.0,2483200.0,2485900.0,2482100.0,2483800.0,2483400.0,2486100.0,2482300.0,2484000.0,2483100.0,2485800.0,2482000.0,2483700.0,169189.0,563.963333,119095.0,396.983333,218189.0,727.296667,188892.0,629.640000,2483200.0,2486000.0,2482100.0,2483900.0,-0.060000,2019-01-02
2019-01-02 10:15:00,2484000.0,2485600.0,2480300.0,2481000.0,2483900.0,2485500.0,2480200.0,2480900.0,2484100.0,2485700.0,2480400.0,2481100.0,2483800.0,2485400.0,2480100.0,2480800.0,134633.0,448.776667,157760.0,525.866667,198805.0,662.683333,168107.0,560.356667,2484000.0,2485600.0,2480200.0,2480900.0,-0.206667,2019-01-02


In [32]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.tail(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2019_corrected.csv', index=False)

Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2019-12-31 15:10:00,3210400.0,3212100.0,3210300.0,3211400.0,3210300.0,3212000.0,3210200.0,3211300.0,3210500.0,3212200.0,3210400.0,3211500.0,3210200.0,3211900.0,3210100.0,3211200.0,302591.0,1039.831615,444801.0,1528.525773,599597.0,2060.470790,587302.0,2018.219931,3210300.0,3212100.0,3210200.0,3211500.0,-0.099656,2019-12-31
2019-12-31 15:15:00,3211500.0,3213200.0,3210400.0,3212200.0,3211300.0,3213100.0,3210300.0,3212100.0,3211600.0,3213300.0,3210500.0,3212300.0,3211200.0,3213000.0,3210200.0,3212000.0,316091.0,1053.636667,378989.0,1263.296667,621087.0,2070.290000,535892.0,1786.306667,3211300.0,3213300.0,3210300.0,3212200.0,-0.053333,2019-12-31
2019-12-31 15:20:00,3212200.0,3213300.0,3212200.0,3212700.0,3212100.0,3213200.0,3212000.0,3212500.0,3212300.0,3213400.0,3212300.0,3212800.0,3212000.0,3213100.0,3211900.0,3212400.0,500586.0,1685.474747,387009.0,1303.060606,979109.0,3296.663300,502673.0,1692.501684,3212200.0,3213400.0,3212000.0,3212600.0,-0.117845,2019-12-31
2019-12-31 15:25:00,3212700.0,3214000.0,3212300.0,3213900.0,3212600.0,3213900.0,3212200.0,3213800.0,3212800.0,3214100.0,3212400.0,3214000.0,3212500.0,3213800.0,3212100.0,3213700.0,477986.0,1603.979866,318898.0,1070.127517,978186.0,3282.503356,512238.0,1718.919463,3212600.0,3214100.0,3212300.0,3213800.0,0.013423,2019-12-31
2019-12-31 15:30:00,3214000.0,3215100.0,3213700.0,3215100.0,3213900.0,3215000.0,3213600.0,3215000.0,3214100.0,3215200.0,3213800.0,3215200.0,3213800.0,3214900.0,3213500.0,3214900.0,600715.0,2009.080268,365009.0,1220.765886,1234428.0,4128.521739,477579.0,1597.254181,3214000.0,3215100.0,3213600.0,3215100.0,0.023411,2019-12-31
2019-12-31 15:35:00,3215100.0,3215100.0,3213800.0,3214600.0,3215000.0,3215000.0,3213700.0,3214500.0,3215200.0,3215200.0,3213900.0,3214700.0,3214900.0,3214900.0,3213600.0,3214400.0,549263.0,1843.164430,293270.0,984.127517,1119203.0,3755.714765,409883.0,1375.446309,3215100.0,3215200.0,3213700.0,3214600.0,-0.147651,2019-12-31
2019-12-31 15:40:00,3214600.0,3216600.0,3214600.0,3216600.0,3214500.0,3216500.0,3214500.0,3216400.0,3214700.0,3216700.0,3214700.0,3216700.0,3214400.0,3216400.0,3214400.0,3216300.0,596165.0,1993.862876,633511.0,2118.765886,1218542.0,4075.391304,614487.0,2055.140468,3214500.0,3216600.0,3214500.0,3216600.0,0.036789,2019-12-31
2019-12-31 15:45:00,3216500.0,3216900.0,3215200.0,3215900.0,3216400.0,3216800.0,3215100.0,3215800.0,3216600.0,3217000.0,3215300.0,3216000.0,3216300.0,3216700.0,3215000.0,3215700.0,567996.0,1893.320000,676523.0,2255.076667,1215071.0,4050.236667,497968.0,1659.893333,3216500.0,3216900.0,3215100.0,3215800.0,0.046667,2019-12-31
2019-12-31 15:50:00,3215800.0,3218500.0,3214300.0,3218400.0,3215700.0,3218400.0,3214200.0,3218300.0,3215900.0,3218600.0,3214400.0,3218500.0,3215600.0,3218300.0,3214100.0,3218200.0,517403.0,1724.676667,757695.0,2525.650000,1006076.0,3353.586667,960120.0,3200.400000,3215700.0,3218600.0,3214200.0,3218200.0,0.126667,2019-12-31
2019-12-31 15:55:00,3218300.0,3221200.0,3216600.0,3220000.0,3218200.0,3221100.0,3216500.0,3219900.0,3218400.0,3221300.0,3216700.0,3220100.0,3218100.0,3221000.0,3216400.0,3219800.0,550142.0,1833.806667,736784.0,2455.946667,814817.0,2716.056667,1401940.0,4673.133333,3218200.0,3221200.0,3216500.0,3219900.0,-0.006667,2019-12-31
